# Mental Health Disorders in Mexico | Extended Data Analysis (2014–2023)

**Data Engineering · Exploratory Analysis · State Comparisons · Temporal Trends · Experimental Forecasting**

**Author:** Data Engineer Luis Barahona  
**Research context:** Student Mental Health, Depression and Suicide Trends in Mexico  
**Notebook:** Extended and corrected version of `Analisis_Trastornos_Mentales.ipynb`

> **Scope:** This notebook analyzes *recorded new-case rates* for selected mental and behavioral disorders by state, sex and year. Rates must not be interpreted as patient-level diagnoses, prevalence, or suicide risk. Confirm the rate denominator and source definitions in the accompanying documentation.

**Expected input:** `mental_salud_limpio.xlsx` — prepared in the companion project and available through the research [Google Drive folder](https://drive.google.com/drive/folders/19m9xrpvTypQDonmSTG4Wks-zC5gT127t?usp=sharing).

## Research objectives and workflow

1. Audit and standardize the Excel dataset, without inventing missing observations.
2. Analyze national trends and compare sex-specific series, when supported by the data.
3. Compare states, disorder categories and geographic regions.
4. Quantify year-over-year changes, temporal variability and missing-data coverage.
5. Perform an **illustrative** historical forecasting backtest, only when the available series is long enough.
6. Export reproducible tables, figures and an automated summary.

**Important:** No numeric findings are predetermined. All tables, charts and conclusions are generated from the uploaded dataset.

## 1. Environment and configuration

In [ ]:
# In Google Colab, uncomment if a dependency is missing:
# %pip install -q pandas numpy matplotlib seaborn openpyxl statsmodels

from pathlib import Path
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 35)

PROJECT_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'mental_health_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_PATH = PROJECT_ROOT / 'mental_salud_limpio.xlsx'
TARGET_DISORDER = 'Depresión'
FINAL_YEAR = 2023
print('Data directory:', PROJECT_ROOT)
print('Output directory:', OUTPUT_DIR)

### 1.1 Load your prepared Excel file

Upload `mental_salud_limpio.xlsx` to Colab using the Files panel, or place it in the current working directory. If it is missing, the next cell asks for an upload in Colab. The notebook deliberately does **not** substitute synthetic data for real observations.

In [ ]:
if not INPUT_PATH.exists():
    try:
        from google.colab import files
        print('Please upload mental_salud_limpio.xlsx')
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith(('.xlsx', '.xls'))]
        if not candidates:
            raise FileNotFoundError('No Excel workbook was uploaded.')
        INPUT_PATH = PROJECT_ROOT / candidates[0]
    except ImportError:
        raise FileNotFoundError(
            f'Missing {INPUT_PATH.name}. Place the prepared Excel file in {PROJECT_ROOT}.'
        )

raw = pd.read_excel(INPUT_PATH)
print('Source:', INPUT_PATH.name)
print('Original shape:', raw.shape)
display(raw.head(5))
print('Columns:', list(raw.columns))

## 2. Standardization and data quality

In [ ]:
def normalize_text(value):
    """Normalize a textual label for matching; leave displayed labels unchanged."""
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', text.strip().lower())

def find_column(frame, candidates):
    mapping = {normalize_text(col): col for col in frame.columns}
    for candidate in candidates:
        if normalize_text(candidate) in mapping:
            return mapping[normalize_text(candidate)]
    return None

def parse_year(col):
    if isinstance(col, (int, np.integer)) and 1900 <= int(col) <= 2100:
        return int(col)
    if isinstance(col, (float, np.floating)) and np.isfinite(col) and col.is_integer() and 1900 <= col <= 2100:
        return int(col)
    value = str(col).strip()
    if re.fullmatch(r'(?:19|20)\d{2}(?:\.0)?', value):
        return int(float(value))
    return None

ENTITY_COL = find_column(raw, ['Entidad Federativa','Entidad','Estado'])
DISORDER_COL = find_column(raw, ['Trastorno','Enfermedad','Padecimiento'])
SEX_COL = find_column(raw, ['Sexo','Género','Genero'])
YEAR_MAP = {col: parse_year(col) for col in raw.columns if parse_year(col) is not None}

if ENTITY_COL is None or DISORDER_COL is None or not YEAR_MAP:
    raise ValueError(
        'Unrecognized data structure. Expected Entidad Federativa, Trastorno, and year columns. '
        f'Found columns: {list(raw.columns)}'
    )
print('Entity column:', ENTITY_COL)
print('Disorder column:', DISORDER_COL)
print('Sex column:', SEX_COL or 'not explicitly present')
print('Year range:', min(YEAR_MAP.values()), '-', max(YEAR_MAP.values()))

frame = raw.rename(columns={ENTITY_COL:'Entidad Federativa', DISORDER_COL:'Trastorno'}).copy()
if SEX_COL is not None:
    frame = frame.rename(columns={SEX_COL:'Sexo'})
else:
    # Original notebook assumes Total / Hombres / Mujeres occupy three rows.
    # Infer this ordering ONLY when every (entity, disorder) group has exactly 3 rows.
    sizes = frame.groupby(['Entidad Federativa','Trastorno'], dropna=False).size()
    if len(sizes) and sizes.eq(3).all():
        frame['Sexo'] = frame.groupby(['Entidad Federativa','Trastorno'],dropna=False).cumcount().map(
            {0:'Total',1:'Hombres',2:'Mujeres'}
        )
        warnings.warn('Sex labels inferred from row order (Total, Hombres, Mujeres) as in original notebook. Verify against the original Excel source.')
    elif len(sizes) and sizes.eq(1).all():
        frame['Sexo'] = 'No especificado'
        warnings.warn('No sex column available: rows are labeled No especificado. Sex comparisons will be skipped.')
    else:
        raise ValueError('No explicit Sexo field and multiple rows per entity/disorder with ambiguous ordering. Review source data before analysis.')

frame['Entidad Federativa'] = frame['Entidad Federativa'].astype('string').str.strip()
frame['Trastorno'] = frame['Trastorno'].astype('string').str.strip()
frame['Sexo'] = frame['Sexo'].astype('string').str.strip()
frame['Sexo'] = frame['Sexo'].map(lambda x: {'total':'Total','hombre':'Hombres','hombres':'Hombres','masculino':'Hombres','mujer':'Mujeres','mujeres':'Mujeres','femenino':'Mujeres'}.get(normalize_text(x),str(x)))

long = frame.melt(
    id_vars=['Entidad Federativa','Trastorno','Sexo'],
    value_vars=list(YEAR_MAP),
    var_name='year_original', value_name='Tasa'
)
long['Año'] = long['year_original'].map(YEAR_MAP).astype(int)
long['Tasa'] = pd.to_numeric(long['Tasa'].astype('string').str.replace(',','.',regex=False),errors='coerce')
long = long.drop(columns=['year_original'])
long = long.dropna(subset=['Entidad Federativa','Trastorno'])
long = long.sort_values(['Trastorno','Entidad Federativa','Sexo','Año']).reset_index(drop=True)

NATIONAL_NAMES = {'estados unidos mexicanos','nacional','mexico','republica mexicana'}
long['Es nacional'] = long['Entidad Federativa'].map(lambda s: normalize_text(s) in NATIONAL_NAMES)
long['Trastorno_normalizado'] = long['Trastorno'].map(normalize_text)
print('Long-form records:',len(long))
print('Disorders:',long['Trastorno'].nunique())
print('Entities:',long['Entidad Federativa'].nunique())
print('Sex labels:', sorted(long['Sexo'].dropna().unique().tolist()))
display(long.head())

In [ ]:
# Data quality audit: never convert unavailable rates to zero.
key_cols = ['Entidad Federativa','Trastorno','Sexo','Año']
duplicates = long.duplicated(subset=key_cols,keep=False)
quality = pd.DataFrame({
    'Metric':['Source rows','Long-format rows','Missing rate cells','Negative rate cells',
              'Duplicate entity/disorder/sex/year keys','Distinct states/entities','Distinct disorders',
              'Available calendar years'],
    'Value':[len(raw),len(long),int(long['Tasa'].isna().sum()),int((long['Tasa']<0).sum()),
             int(duplicates.sum()),long['Entidad Federativa'].nunique(),long['Trastorno'].nunique(),long['Año'].nunique()]
})
display(quality)
if duplicates.any():
    display(long.loc[duplicates,key_cols+['Tasa']].head(15))
    raise ValueError('Duplicate entity/disorder/sex/year keys found. Resolve ambiguity at the source; do not average or sum rates automatically.')
if (long['Tasa']<0).any():
    warnings.warn('Negative rates detected: verify source or transformation rules.')
if not long['Es nacional'].any():
    warnings.warn('No national row identified. National analyses will be skipped; state rates must not be summed to obtain a national rate.')

coverage = long.assign(Disponibilidad=long['Tasa'].notna()).groupby(
    ['Año','Sexo'],dropna=False)['Disponibilidad'].agg(['sum','count']).reset_index()
coverage['% disponible']=100*coverage['sum']/coverage['count']
display(coverage.head(15))

## 3. Analytical definitions

This notebook distinguishes `Total`, `Hombres` and `Mujeres`. When a source lacks an explicit sex field, its values must be confirmed before interpreting sex-specific results. National rates are taken from **national rows**, not calculated by summing state rates. Missing values remain missing.

**Regional rates** below are unweighted arithmetic averages of available state rates, **not population-weighted regional incidence**. State comparisons are descriptive and may depend on reporting practices and differences in denominators.

In [ ]:
disorders = sorted(long['Trastorno'].dropna().unique().tolist())
match = [x for x in disorders if normalize_text(x)==normalize_text(TARGET_DISORDER)]
if not match:
    match = [x for x in disorders if 'depres' in normalize_text(x)]
if not match:
    print('Depression not found; using the first available disorder for exploratory examples.')
    selected_disorder = disorders[0]
else:
    selected_disorder=match[0]
print('Selected disorder:',selected_disorder)

def pick_total(frame):
    options = frame.loc[frame['Sexo'].eq('Total')]
    if not options.empty: return options
    options = frame.loc[frame['Sexo'].eq('No especificado')]
    return options

def saveplot(stem):
    plt.tight_layout()
    path=OUTPUT_DIR / f'{stem}.png'
    plt.savefig(path,dpi=150,bbox_inches='tight')
    plt.show()
    return path

## 4. National depression trends (original section 1, corrected)

In [ ]:
national = long[long['Es nacional'] & long['Trastorno'].eq(selected_disorder)].copy()
national_total = pick_total(national).dropna(subset=['Tasa']).sort_values('Año')
if national_total.empty:
    print('National total series unavailable; no national trend chart generated.')
else:
    display(national_total[['Año','Tasa']])
    plt.figure(figsize=(11,4.5))
    sns.lineplot(data=national_total,x='Año',y='Tasa',marker='o')
    plt.title(f'National recorded new-case rate — {selected_disorder}')
    plt.ylabel('Reported rate (confirm denominator)')
    saveplot('01_national_depression_trend')

## 5. National rates by sex (original section 2, without duplicate plotting)

In [ ]:
sex_national=national[national['Sexo'].isin(['Hombres','Mujeres'])].dropna(subset=['Tasa'])
if sex_national.empty:
    print('Sex-specific national data unavailable; skipping comparison.')
else:
    comparison=sex_national.pivot(index='Año',columns='Sexo',values='Tasa').sort_index()
    display(comparison)
    plt.figure(figsize=(11,4.5))
    sns.lineplot(data=sex_national,x='Año',y='Tasa',hue='Sexo',marker='o')
    plt.title(f'National recorded rate by sex — {selected_disorder}')
    plt.ylabel('Reported rate (confirm denominator)')
    saveplot('02_depression_by_sex')
    if {'Hombres','Mujeres'}.issubset(comparison.columns):
        sex_gap=comparison.assign(Difference_W_minus_M=comparison['Mujeres']-comparison['Hombres'])
        display(sex_gap.tail())

## 6. Top states in the most recent available year (original section 3)

In [ ]:
state_data = pick_total(long[(~long['Es nacional']) & long['Trastorno'].eq(selected_disorder)]).copy()
state_available=state_data.dropna(subset=['Tasa'])
if state_available.empty:
    print('State-level total rates are unavailable.')
else:
    latest_year=min(FINAL_YEAR,int(state_available['Año'].max()))
    state_current=state_available[state_available['Año'].eq(latest_year)]
    top_states=state_current.nlargest(10,'Tasa')[['Entidad Federativa','Tasa']]
    print('Comparison year:',latest_year)
    display(top_states)
    plt.figure(figsize=(10,5.5))
    sns.barplot(data=top_states,x='Tasa',y='Entidad Federativa',color='#bf5252')
    plt.title(f'Top states by recorded {selected_disorder} rate — {latest_year}')
    plt.xlabel('Reported rate (confirm denominator)')
    saveplot('03_top_states')

## 7. National comparison across disorders (original section 4)

In [ ]:
national_all = pick_total(long[long['Es nacional']]).dropna(subset=['Tasa'])
if national_all.empty:
    print('No comparable national-total disorder rows.')
else:
    year_national=min(FINAL_YEAR,int(national_all['Año'].max()))
    disorder_comparison=(national_all[national_all['Año'].eq(year_national)]
       .sort_values('Tasa',ascending=False)[['Trastorno','Tasa']])
    display(disorder_comparison)
    plt.figure(figsize=(10,max(4,0.48*len(disorder_comparison))))
    sns.barplot(data=disorder_comparison,x='Tasa',y='Trastorno',color='#3c83af')
    plt.title(f'National recorded rates by disorder — {year_national}')
    saveplot('04_national_disorders')

## 8. Regional exploratory analysis (original section 5, improved geographic coverage)

In [ ]:
REGIONS = {
 'Northwest':['Baja California','Baja California Sur','Sonora','Sinaloa','Chihuahua'],
 'Northeast':['Coahuila','Nuevo León','Tamaulipas','Durango'],
 'West':['Jalisco','Colima','Michoacán','Nayarit','Zacatecas','Aguascalientes'],
 'Central':['Ciudad de México','México','Estado de México','Hidalgo','Puebla','Tlaxcala','Morelos','Querétaro','Guanajuato','San Luis Potosí'],
 'South':['Guerrero','Oaxaca','Chiapas'],
 'Southeast':['Veracruz','Tabasco','Campeche','Yucatán','Quintana Roo']
}
STATE_TO_REGION={normalize_text(st):reg for reg,states in REGIONS.items() for st in states}
if 'state_current' in globals() and not state_current.empty:
    regional=state_current.copy()
    regional['Region']=regional['Entidad Federativa'].map(lambda x:STATE_TO_REGION.get(normalize_text(x),'Unmapped'))
    unmapped=sorted(regional.loc[regional['Region']=='Unmapped','Entidad Federativa'].unique())
    if unmapped: print('Review unmapped entity names:',unmapped)
    regional_summary=regional.groupby('Region',as_index=False).agg(
        Mean_rate=('Tasa','mean'),States_with_data=('Tasa','count'),Median_rate=('Tasa','median'))
    display(regional_summary.sort_values('Mean_rate',ascending=False))
    plt.figure(figsize=(10,4.5))
    sns.barplot(data=regional_summary.sort_values('Mean_rate',ascending=False),x='Mean_rate',y='Region',color='#429a82')
    plt.title(f'Unweighted mean of available state rates — {latest_year}')
    plt.xlabel('Arithmetic mean of recorded state rates (NOT a regional incidence rate)')
    saveplot('05_region_unweighted_means')

## 9. Highest recorded national years (original section 6)

In [ ]:
if not national_total.empty:
    critical_years=national_total.nlargest(5,'Tasa')[['Año','Tasa']]
    display(critical_years)
    print('Descriptive ranking only: high recorded values do not independently establish a public-health emergency.')

## 10. New analyses — annual change and temporal variability

Year-over-year differences quantify movement in the recorded rate, not changes in the number of affected people. Large percentage changes can be unstable when the previous rate is close to zero.

In [ ]:
if not national_total.empty:
    national_changes=national_total[['Año','Tasa']].sort_values('Año').copy()
    national_changes['Absolute change']=national_changes['Tasa'].diff()
    national_changes['Change (%)']=national_changes['Tasa'].pct_change(fill_method=None)*100
    display(national_changes)
    plt.figure(figsize=(11,4.5))
    sns.barplot(data=national_changes,x='Año',y='Absolute change',color='#5685aa')
    plt.axhline(0,color='black',linewidth=0.8)
    plt.xticks(rotation=45)
    plt.title(f'Year-over-year absolute change — {selected_disorder}')
    saveplot('06_national_annual_changes')

## 11. New analysis — state-by-year heatmap

In [ ]:
if not state_available.empty:
    heat=state_available.pivot(index='Entidad Federativa',columns='Año',values='Tasa')
    heat=heat.loc[heat.mean(axis=1,skipna=True).sort_values(ascending=False).index].astype(float)
    plt.figure(figsize=(12,max(6,len(heat)*0.28)))
    sns.heatmap(heat,cmap='YlOrRd',linewidths=.15,mask=heat.isna(),cbar_kws={'label':'Reported rate'})
    plt.title(f'State-by-year rates — {selected_disorder} (blank = missing)')
    saveplot('07_state_year_heatmap')
    coverage_states=heat.notna().sum(axis=1).rename('Years_with_data').reset_index()
    display(coverage_states.sort_values('Years_with_data').head(10))

## 12. New analysis — distribution and outlier review

In [ ]:
if not state_available.empty:
    subset=state_available[state_available['Año'].eq(latest_year)].copy()
    if len(subset)>=4:
        q1,q3=subset['Tasa'].quantile([.25,.75]); iqr=q3-q1
        lower,upper=q1-1.5*iqr,q3+1.5*iqr
        flagged=subset[(subset['Tasa']<lower)|(subset['Tasa']>upper)][['Entidad Federativa','Tasa']]
        print('IQR bounds:',round(lower,4),round(upper,4))
        print('Outliers to review:',len(flagged))
        display(flagged.sort_values('Tasa',ascending=False))
        plt.figure(figsize=(9,3.7))
        sns.boxplot(x=subset['Tasa'],color='#77b8aa')
        plt.title(f'State rate distribution — {latest_year} (statistical outliers ≠ errors)')
        saveplot('08_state_distribution')

## 13. New analysis — historical holdout forecast (illustrative)

A short, transparent holdout experiment is preferable to reporting unvalidated predictions. If the national series contains at least **seven consecutive annual observations**, the last two years are withheld for evaluation. Models: last-observed-value baseline and Holt's additive trend method. The data are never split randomly. A best model is selected by holdout MAE only as a teaching exercise; two observations are insufficient for dependable model selection.

**Forecasts are statistical extrapolations, not medical or policy recommendations.**

In [ ]:
forecast_results=pd.DataFrame()
future_forecast=pd.DataFrame()
if national_total.empty:
    print('Forecast skipped: national total series unavailable.')
else:
    ts=national_total.set_index('Año')['Tasa'].sort_index()
    expected_years=np.arange(ts.index.min(),ts.index.max()+1)
    ts=ts.reindex(expected_years)
    if len(ts)<7 or ts.isna().any():
        print('Forecast skipped: requires >=7 consecutive non-missing annual rates. Available:',len(ts),'Missing:',int(ts.isna().sum()))
    else:
        from statsmodels.tsa.holtwinters import Holt
        from sklearn.metrics import mean_absolute_error, mean_squared_error
        train,test=ts.iloc[:-2],ts.iloc[-2:]
        baseline_pred=np.repeat(train.iloc[-1],len(test))
        preds={'Persistence':baseline_pred}
        try:
            fit_holt=Holt(train.astype(float),damped_trend=True,initialization_method='estimated').fit(optimized=True)
            preds['Damped Holt']=np.asarray(fit_holt.forecast(len(test)),dtype=float)
        except Exception as ex:
            print('Holt could not be fitted; baseline remains available:',ex)
        metrics=[]
        for name,pred in preds.items():
            metrics.append({'Model':name,'MAE':mean_absolute_error(test,pred),
                            'RMSE':np.sqrt(mean_squared_error(test,pred)),'Holdout years':', '.join(map(str,test.index))})
        forecast_results=pd.DataFrame(metrics).sort_values('MAE').reset_index(drop=True)
        display(forecast_results)
        best=forecast_results.loc[0,'Model']
        horizon=3
        forecast_years=np.arange(int(ts.index.max())+1,int(ts.index.max())+horizon+1)
        if best=='Persistence':
            values=np.repeat(ts.iloc[-1],horizon)
        else:
            full_fit=Holt(ts.astype(float),damped_trend=True,initialization_method='estimated').fit(optimized=True)
            values=np.asarray(full_fit.forecast(horizon),dtype=float)
        future_forecast=pd.DataFrame({'Año':forecast_years,'Exploratory forecast':np.maximum(0,values),'Model':best})
        display(future_forecast)
        plt.figure(figsize=(11,4.5))
        plt.plot(ts.index,ts.values,'o-',label='Observed national rate')
        plt.plot(forecast_years,future_forecast['Exploratory forecast'],'o--',label=f'{best} extrapolation')
        plt.axvline(ts.index.max(),ls=':',color='gray')
        plt.title('Historical rates and illustrative extrapolation — not validated predictions')
        plt.ylabel('Reported rate (confirm denominator)')
        plt.legend()
        saveplot('09_illustrative_forecast')

## 14. New analysis — automated descriptive conclusions

In [ ]:
report_lines=[
    '## Automatically generated summary (based only on available source data)',
    f'- Source workbook: **{INPUT_PATH.name}**.',
    f'- Available years: **{long["Año"].min()}–{long["Año"].max()}**.',
    f'- Distinct disorder labels: **{long["Trastorno"].nunique()}**.',
    f'- Missing rate cells: **{int(long["Tasa"].isna().sum())}**.',
    f'- Selected disorder: **{selected_disorder}**.'
]
if not national_total.empty:
    first,last=national_total.iloc[0],national_total.iloc[-1]
    report_lines.append(f'- National total recorded rate changes from **{first.Tasa:.3f} ({int(first.Año)})** to **{last.Tasa:.3f} ({int(last.Año)})**; this is a descriptive comparison, not causal evidence.')
if 'top_states' in globals() and not top_states.empty:
    report_lines.append(f'- Highest listed state in {latest_year}: **{top_states.iloc[0]["Entidad Federativa"]}** (recorded rate **{top_states.iloc[0]["Tasa"]:.3f}**).')
if not forecast_results.empty:
    report_lines.append(f'- Best two-year holdout MAE among evaluated teaching models: **{forecast_results.iloc[0]["Model"]}**, MAE **{forecast_results.iloc[0]["MAE"]:.3f}**. This is not an operational validation.')
report_lines.append('- **Interpretation limits:** observational data, reporting variation, potential missing years and unknown source denominator; no individual diagnosis or prediction of suicide risk.')
summary='\n'.join(report_lines)
display(Markdown(summary))
(OUTPUT_DIR/'summary.md').write_text(summary,encoding='utf-8')

## 15. Export tables and figures

Exports only **derived aggregate data** from this notebook. The original source file is not modified. Check licenses and publication permissions before sharing any output. Avoid publishing raw student survey records or identifying information from the larger research project.

In [ ]:
tables={
 'data_quality':quality,
 'tidy_rates':long.drop(columns=['Trastorno_normalizado','Es nacional']),
 'coverage':coverage,
 'national_selected_disorder':national_total.drop(columns=['Trastorno_normalizado','Es nacional'],errors='ignore'),
 'states_selected_disorder':state_available.drop(columns=['Trastorno_normalizado','Es nacional'],errors='ignore'),
 'forecast_holdout_metrics':forecast_results,
 'illustrative_forecast':future_forecast
}
for name,data in tables.items():
    if isinstance(data,pd.DataFrame) and not data.empty:
        data.to_csv(OUTPUT_DIR / f'{name}.csv',index=False,encoding='utf-8-sig')

xlsx_path=OUTPUT_DIR/'mental_health_analysis_tables.xlsx'
with pd.ExcelWriter(xlsx_path,engine='openpyxl') as writer:
    for name,data in tables.items():
        if isinstance(data,pd.DataFrame) and not data.empty:
            data.to_excel(writer,sheet_name=name[:31],index=False)
print('Reports exported to:',OUTPUT_DIR)
for file in sorted(OUTPUT_DIR.iterdir()):
    print('-',file.name)

## 16. Final interpretation and research integrity

**What this analysis can support:** descriptions of the recorded rates available in the source, differences by year, sex and state, coverage diagnostics, and comparisons of simple forecasting baselines.

**What it cannot establish:** individual mental health status, true population prevalence without verified denominators, causal drivers of disorders, individual suicide risk, or clinical effectiveness of interventions. Differences in reporting, surveillance, case definitions and population denominators may contribute to observed patterns.

**Known limitations of the original 15-cell notebook addressed here:** brittle year-column detection; positional assumptions about sex rows; duplicate sex-comparison chart; potential ambiguity from multiple records per disorder/state; implicit exclusion of missing values; and regional calculations labeled clearly as unweighted averages.

**Recommended next research step:** obtain the official source dictionary for `mental_salud_limpio.xlsx`, confirm whether the rows represent `Total`, `Hombres` and `Mujeres`, verify the rate denominator, and document coverage of all federal entities before publishing analytical claims.

---

### Research material

[Project data and supporting references (Google Drive)](https://drive.google.com/drive/folders/19m9xrpvTypQDonmSTG4Wks-zC5gT127t?usp=sharing)

### Contact

**Data Engineer Luis Barahona**

| Platform | Contact |
|:---|:---|
| Email | [190300395@ucaribe.edu.mx](mailto:190300395@ucaribe.edu.mx) |
| GitHub | [github.com/LuisBarahonaMX](https://github.com/LuisBarahonaMX) |
| LinkedIn | [linkedin.com/in/luis-barahona-mx](https://www.linkedin.com/in/luis-barahona-mx/) |
| Instagram | [@luisbarahonamx](https://www.instagram.com/luisbarahonamx/) |
| X | [@luisbarahonaMX](https://x.com/luisbarahonaMX) |
| Website | [SIKO AI Platform](https://sikoai.web.app/) |